In [ ]:
# Capital Budgeting — Project Selection (Alias-Based Model)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()

# -------------------------------------------------------------------------------
# Problem Data (synthetic investment portfolio)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(41)

n_projects = rng.integers(20, 50)

# Project costs and expected returns
costs = rng.integers(10, 80, size=n_projects)
returns = rng.integers(15, 100, size=n_projects)

# Budget set as fraction of total cost (controls tightness)
budget = int(0.4 * np.sum(costs))

print(f"Backend:  {m.backend_name}")
print(f"Projects: {n_projects}")
print(f"Costs:    {costs}")
print(f"Returns:  {returns}")
print(f"Budget:   {budget}")

# -------------------------------------------------------------------------------
# Decision Variables
# -------------------------------------------------------------------------------
x = [m.add_binary_var(name=f"x_{j}") for j in range(n_projects)]

# -------------------------------------------------------------------------------
# Constraint (budget)
# -------------------------------------------------------------------------------
m.add_constraint(
    quicksum(costs[j] * x[j] for j in range(n_projects)) <= budget
)

# -------------------------------------------------------------------------------
# Objective (maximize return)
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(returns[j] * x[j] for j in range(n_projects)),
    GRB.MAXIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
solution = np.array([v.x for v in x])

selected = np.where(solution > 0.5)[0]

total_cost = np.sum(costs * solution)
total_return = np.sum(returns * solution)

print("\nSelected projects:", selected)
print("Total cost:", total_cost)
print("Total return:", total_return)
print("Budget used:", total_cost, "/", budget)